# Finale Optimierung mit besten Hyperparametern

Dieses Notebook nimmt die besten Hyperparameter aus dem Tuning und führt eine **intensive finale Optimierung** durch:

- Höhere Anzahl Simulationen (R)
- Mehr Generationen
- Mehrere Seeds für Robustheit
- Kombinierte Pareto-Front aus allen Runs

---

## 1. Setup

In [1]:
import numpy as np
import random
import time
import pandas as pd
import matplotlib.pyplot as plt
import json
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# ============================================================
# WARTE AUF TUNING-ERGEBNISSE (falls noch nicht fertig)
# ============================================================
RESULTS_DIR = Path("../results")
TUNING_FILES = [
    RESULTS_DIR / 'parallel_tuning_results.json',
    RESULTS_DIR / 'best_hyperparameters.json',
    RESULTS_DIR / 'full_tuning_results.json'
]

CHECK_INTERVAL = 60  # Sekunden zwischen Checks
MAX_WAIT_HOURS = 12  # Maximale Wartezeit

def wait_for_tuning():
    """Wartet bis das Hyperparameter-Tuning abgeschlossen ist."""
    start_wait = time.time()
    max_wait = MAX_WAIT_HOURS * 3600
    
    while True:
        # Prüfe ob eine der Ergebnis-Dateien existiert
        for f in TUNING_FILES:
            if f.exists():
                # Prüfe ob Datei vollständig ist (nicht gerade geschrieben wird)
                try:
                    with open(f, 'r') as file:
                        data = json.load(file)
                    if 'best_per_algorithm' in data or any(k in data for k in ['NSGA-II', 'SMS-EMOA', 'NSGA-III']):
                        print(f"\n✓ Tuning-Ergebnisse gefunden: {f}")
                        print(f"  Gewartet: {(time.time() - start_wait)/60:.1f} Minuten")
                        return True
                except (json.JSONDecodeError, KeyError):
                    pass  # Datei noch nicht vollständig
        
        # Timeout prüfen
        elapsed = time.time() - start_wait
        if elapsed > max_wait:
            print(f"\n✗ Timeout nach {MAX_WAIT_HOURS} Stunden")
            return False
        
        # Status anzeigen
        now = datetime.now().strftime("%H:%M:%S")
        waited = elapsed / 60
        print(f"[{now}] Warte auf Tuning-Ergebnisse... ({waited:.0f} min)", end='\r')
        
        time.sleep(CHECK_INTERVAL)

# Warte auf Ergebnisse
print("=" * 60)
print("WARTE AUF HYPERPARAMETER-TUNING")
print("=" * 60)
print(f"Prüfe alle {CHECK_INTERVAL} Sekunden auf: {[f.name for f in TUNING_FILES]}")
print("Starte automatisch sobald Ergebnisse vorliegen...\n")

if not any(f.exists() for f in TUNING_FILES):
    tuning_ready = wait_for_tuning()
    if not tuning_ready:
        raise RuntimeError("Tuning-Ergebnisse nicht gefunden!")
else:
    print("✓ Tuning-Ergebnisse bereits vorhanden!")

print("\n" + "=" * 60)
print("STARTE FINALE OPTIMIERUNG")
print("=" * 60 + "\n")

# ============================================================
# IMPORTS
# ============================================================
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.nsga3 import NSGA3
from pymoo.algorithms.moo.sms import SMSEMOA
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize
from pymoo.termination import get_termination
from pymoo.indicators.hv import HV
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

from simulation import TopTrumpsSimulation, TopTrumpsBalancing

# Matplotlib Style
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    try:
        plt.style.use('seaborn-whitegrid')
    except:
        pass

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("Setup complete")

WARTE AUF HYPERPARAMETER-TUNING
Prüfe alle 60 Sekunden auf: ['parallel_tuning_results.json', 'best_hyperparameters.json', 'full_tuning_results.json']
Starte automatisch sobald Ergebnisse vorliegen...

✓ Tuning-Ergebnisse bereits vorhanden!

STARTE FINALE OPTIMIERUNG

Setup complete


In [2]:
# Verzeichnisse (RESULTS_DIR bereits oben definiert)
PLOTS_DIR = Path("../plots")
RESULTS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

---

## 2. Beste Hyperparameter laden

In [3]:
# Versuche verschiedene Quellen für beste Parameter
best_params = None

# Option 1: Aus parallel_tuning_results.json
try:
    with open(RESULTS_DIR / 'parallel_tuning_results.json', 'r') as f:
        tuning_results = json.load(f)
    best_params = tuning_results.get('best_per_algorithm', {})
    print("Loaded from: parallel_tuning_results.json")
except FileNotFoundError:
    pass

# Option 2: Aus best_hyperparameters.json
if not best_params:
    try:
        with open(RESULTS_DIR / 'best_hyperparameters.json', 'r') as f:
            best_params = json.load(f)
        print("Loaded from: best_hyperparameters.json")
    except FileNotFoundError:
        pass

# Option 3: Aus full_tuning_results.json
if not best_params:
    try:
        with open(RESULTS_DIR / 'full_tuning_results.json', 'r') as f:
            tuning_results = json.load(f)
        best_params = tuning_results.get('best_per_algorithm', {})
        print("Loaded from: full_tuning_results.json")
    except FileNotFoundError:
        pass

# Fallback: Default-Parameter
if not best_params:
    print("WARNING: No tuning results found! Using default parameters.")
    best_params = {
        'NSGA-II': {
            'pop_size': 100,
            'eta_crossover': 15.0,
            'eta_mutation': 20.0,
            'crossover_prob': 0.9
        },
        'SMS-EMOA': {
            'pop_size': 100,
            'eta_crossover': 15.0,
            'eta_mutation': 20.0,
            'crossover_prob': 0.9
        },
        'NSGA-III': {
            'pop_size': 100,
            'eta_crossover': 15.0,
            'eta_mutation': 20.0,
            'crossover_prob': 0.9,
            'n_partitions': 12
        }
    }

print("\n" + "=" * 60)
print("BESTE HYPERPARAMETER")
print("=" * 60)
for algo, params in best_params.items():
    print(f"\n{algo}:")
    for k, v in params.items():
        if k not in ['hypervolume', 'hypervolume_mean', 'hypervolume_std', 'status', 'trial_id', 'algorithm']:
            print(f"  {k}: {v}")

Loaded from: parallel_tuning_results.json

BESTE HYPERPARAMETER

NSGA-II:
  pop_size: 230
  eta_crossover: 18.557523580101787
  eta_mutation: 16.672067054749483
  crossover_prob: 0.7806796324592887
  hypervolume_min: 4.140762
  hypervolume_max: 4.262678
  runtime_seconds: 3641.215940952301
  n_partitions: nan

SMS-EMOA:
  pop_size: 230
  eta_crossover: 19.14143753867365
  eta_mutation: 25.860265505247146
  crossover_prob: 0.9627414011273695
  hypervolume_min: 4.002927000000001
  hypervolume_max: 4.189909
  runtime_seconds: 3465.8906543254852
  n_partitions: nan

NSGA-III:
  pop_size: 140
  eta_crossover: 6.3753416091134705
  eta_mutation: 19.863306691118574
  crossover_prob: 0.8761412941598613
  hypervolume_min: 3.8684860000000003
  hypervolume_max: 4.0499339999999995
  runtime_seconds: 2079.2468690872192
  n_partitions: 17.0


---

## 3. Finale Optimierungs-Konfiguration

In [4]:
# Problem-Parameter
try:
    with open(RESULTS_DIR / 'config.json', 'r') as f:
        CONFIG = json.load(f)
    SEED = CONFIG['seed']
    K = CONFIG['K']
    L = CONFIG['L']
    XL = CONFIG['xl']
    XU = CONFIG['xu']
except FileNotFoundError:
    SEED = 42
    K = 22
    L = 4
    XL = 1.0
    XU = 10.0

# ============================================================
# FINALE OPTIMIERUNGS-PARAMETER (INTENSIV)
# ============================================================

R_FINAL = 2500           # Hohe Anzahl Simulationen für genaue Fitness
N_GEN_FINAL = 200        # Viele Generationen für gute Konvergenz
N_SEEDS = 10             # Mehrere unabhängige Runs
FINAL_SEEDS = list(range(42, 42 + N_SEEDS))  # [42, 43, ..., 51]

# Automatisch besten Algorithmus aus Tuning-Ergebnissen wählen
def get_best_algorithm(params):
    """Wählt den Algorithmus mit dem höchsten Hypervolume."""
    best_algo = None
    best_hv = -1
    
    for algo, p in params.items():
        hv = p.get('hypervolume', p.get('hypervolume_mean', 0))
        if hv and hv > best_hv:
            best_hv = hv
            best_algo = algo
    
    return best_algo if best_algo else 'NSGA-II'

FINAL_ALGORITHM = get_best_algorithm(best_params)

print("=" * 60)
print("FINALE OPTIMIERUNGS-KONFIGURATION")
print("=" * 60)
print(f"Algorithm: {FINAL_ALGORITHM} (automatisch gewählt)")
print(f"Simulations (R): {R_FINAL}")
print(f"Generations: {N_GEN_FINAL}")
print(f"Independent Runs: {N_SEEDS}")
print(f"Problem: K={K}, L={L}")
print("=" * 60)

FINALE OPTIMIERUNGS-KONFIGURATION
Algorithm: NSGA-II (automatisch gewählt)
Simulations (R): 2500
Generations: 200
Independent Runs: 10
Problem: K=22, L=4


In [5]:
def create_algorithm(algo_name, params):
    """Erstellt den Algorithmus mit den gegebenen Hyperparametern."""
    pop_size = int(params.get('pop_size', 100))
    eta_crossover = float(params.get('eta_crossover', 15.0))
    eta_mutation = float(params.get('eta_mutation', 20.0))
    crossover_prob = float(params.get('crossover_prob', 0.9))
    
    if algo_name == 'NSGA-II':
        return NSGA2(
            pop_size=pop_size,
            sampling=FloatRandomSampling(),
            crossover=SBX(prob=crossover_prob, eta=eta_crossover),
            mutation=PM(eta=eta_mutation),
            eliminate_duplicates=True
        )
    elif algo_name == 'SMS-EMOA':
        return SMSEMOA(
            pop_size=pop_size,
            sampling=FloatRandomSampling(),
            crossover=SBX(prob=crossover_prob, eta=eta_crossover),
            mutation=PM(eta=eta_mutation),
            eliminate_duplicates=True
        )
    elif algo_name == 'NSGA-III':
        n_partitions = int(params.get('n_partitions', 12))
        ref_dirs = get_reference_directions("das-dennis", 2, n_partitions=n_partitions)
        return NSGA3(
            pop_size=pop_size,
            ref_dirs=ref_dirs,
            sampling=FloatRandomSampling(),
            crossover=SBX(prob=crossover_prob, eta=eta_crossover),
            mutation=PM(eta=eta_mutation),
            eliminate_duplicates=True
        )
    else:
        raise ValueError(f"Unknown algorithm: {algo_name}")

---

## 4. Finale Optimierung durchführen

In [6]:
# Simulation und Problem erstellen
sim = TopTrumpsSimulation(num_cards=K, num_categories=L)
problem = TopTrumpsBalancing(sim, n_simulations=R_FINAL, xl=XL, xu=XU)

# WICHTIG: termination wird IN der Loop erstellt (nicht hier!)
# Termination-Objekte speichern internen Zustand und dürfen nicht wiederverwendet werden

# Hyperparameter für gewählten Algorithmus
algo_params = best_params.get(FINAL_ALGORITHM, best_params.get(list(best_params.keys())[0]))

print(f"\nUsing {FINAL_ALGORITHM} with parameters:")
for k, v in algo_params.items():
    if k not in ['hypervolume', 'hypervolume_mean', 'hypervolume_std', 'status', 'trial_id', 'algorithm']:
        print(f"  {k}: {v}")


Using NSGA-II with parameters:
  pop_size: 230
  eta_crossover: 18.557523580101787
  eta_mutation: 16.672067054749483
  crossover_prob: 0.7806796324592887
  hypervolume_min: 4.140762
  hypervolume_max: 4.262678
  runtime_seconds: 3641.215940952301
  n_partitions: nan


In [7]:
print("=" * 60)
print("STARTING FINAL OPTIMIZATION")
print("=" * 60)
print(f"Running {N_SEEDS} independent optimizations...")
print("This will take a while.\n")

all_runs = []
all_F = []  # Alle Objective-Werte
all_X = []  # Alle Lösungen
failed_seeds = []

total_start = time.time()

for i, seed in enumerate(FINAL_SEEDS):
    np.random.seed(seed)
    random.seed(seed)
    
    start_time = time.time()
    
    try:
        # Algorithmus und Termination für JEDEN Run neu erstellen
        # (Termination-Objekte speichern internen Zustand!)
        algorithm = create_algorithm(FINAL_ALGORITHM, algo_params)
        termination = get_termination("n_gen", N_GEN_FINAL)
        res = minimize(problem, algorithm, termination, seed=seed, verbose=False)
        
        runtime = time.time() - start_time
        
        # Hypervolume berechnen
        hv = HV(ref_point=np.array([0.0, 0.0]))(res.F)
        
        # Ergebnisse speichern
        run_result = {
            'seed': seed,
            'hypervolume': hv,
            'n_solutions': len(res.F),
            'runtime_seconds': runtime,
            'F': res.F.tolist(),
            'X': res.X.tolist()
        }
        all_runs.append(run_result)
        
        # Für kombinierte Front
        all_F.append(res.F)
        all_X.append(res.X)
        
        # Progress
        elapsed = time.time() - total_start
        completed = len(all_runs)
        remaining = N_SEEDS - i - 1
        eta = elapsed / completed * remaining if completed > 0 else 0
        print(f"[{i+1:2d}/{N_SEEDS}] Seed {seed}: HV={hv:.4f}, Solutions={len(res.F)}, "
              f"Time={runtime/60:.1f}min | Total: {elapsed/60:.1f}min | ETA: {eta/60:.1f}min")
    
    except Exception as e:
        runtime = time.time() - start_time
        failed_seeds.append(seed)
        print(f"[{i+1:2d}/{N_SEEDS}] Seed {seed}: FAILED after {runtime/60:.1f}min - {str(e)[:50]}")

total_time = time.time() - total_start

print("\n" + "=" * 60)
print("FINAL OPTIMIZATION COMPLETED!")
print("=" * 60)
print(f"Total Runtime: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
print(f"Successful Runs: {len(all_runs)}/{N_SEEDS}")
if failed_seeds:
    print(f"Failed Seeds: {failed_seeds}")

# Prüfen ob genug Runs erfolgreich waren
if len(all_runs) < 3:
    raise RuntimeError(f"Too few successful runs ({len(all_runs)}). Cannot continue.")

STARTING FINAL OPTIMIZATION
Running 10 independent optimizations...
This will take a while.

[ 1/10] Seed 42: HV=4.1122, Solutions=73, Time=42.0min | Total: 42.0min | ETA: 378.0min


KeyboardInterrupt: 

---

## 5. Kombinierte Pareto-Front

In [ ]:
# Alle Lösungen kombinieren
combined_F = np.vstack(all_F)
combined_X = np.vstack(all_X)

print(f"Total solutions from all runs: {len(combined_F)}")

# Non-dominated Sorting um beste kombinierte Front zu finden
nds = NonDominatedSorting()
fronts = nds.do(combined_F, only_non_dominated_front=True)

# Beste Front extrahieren
best_F = combined_F[fronts]
best_X = combined_X[fronts]

# Nach Fairness sortieren
sort_idx = np.argsort(best_F[:, 0])[::-1]  # Absteigend nach Fairness
best_F = best_F[sort_idx]
best_X = best_X[sort_idx]

print(f"Combined Pareto-Front: {len(best_F)} non-dominated solutions")

# Hypervolume der kombinierten Front
combined_hv = HV(ref_point=np.array([0.0, 0.0]))(best_F)
print(f"Combined Hypervolume: {combined_hv:.4f}")

---

## 6. Ergebnis-Analyse

In [ ]:
# Statistiken über alle Runs
hvs = [r['hypervolume'] for r in all_runs]
n_sols = [r['n_solutions'] for r in all_runs]
runtimes = [r['runtime_seconds'] for r in all_runs]

print("=" * 60)
print("STATISTIKEN ÜBER ALLE RUNS")
print("=" * 60)
print(f"\nHypervolume:")
print(f"  Mean: {np.mean(hvs):.4f}")
print(f"  Std:  {np.std(hvs):.4f}")
print(f"  Min:  {np.min(hvs):.4f}")
print(f"  Max:  {np.max(hvs):.4f}")

print(f"\nAnzahl Lösungen:")
print(f"  Mean: {np.mean(n_sols):.1f}")
print(f"  Std:  {np.std(n_sols):.1f}")

print(f"\nRuntime pro Run:")
print(f"  Mean: {np.mean(runtimes)/60:.1f} min")

print(f"\nKombinierte Front:")
print(f"  Hypervolume: {combined_hv:.4f}")
print(f"  Lösungen: {len(best_F)}")

In [ ]:
# Extrempunkte der kombinierten Front
print("\n" + "=" * 60)
print("EXTREMPUNKTE DER PARETO-FRONT")
print("=" * 60)

# Beste Fairness
best_fairness_idx = np.argmax(best_F[:, 0])
print(f"\nBeste Fairness (Win Rate):")
print(f"  Fairness: {best_F[best_fairness_idx, 0]:.4f}")
print(f"  Excitement: {best_F[best_fairness_idx, 1]:.4f}")

# Beste Excitement
best_excitement_idx = np.argmax(best_F[:, 1])
print(f"\nBeste Excitement:")
print(f"  Fairness: {best_F[best_excitement_idx, 0]:.4f}")
print(f"  Excitement: {best_F[best_excitement_idx, 1]:.4f}")

# Knee-Point (balanciert)
# Einfache Heuristik: normalisierte Summe maximieren
f_norm = (best_F - best_F.min(axis=0)) / (best_F.max(axis=0) - best_F.min(axis=0) + 1e-10)
knee_idx = np.argmax(f_norm.sum(axis=1))
print(f"\nKnee-Point (balanciert):")
print(f"  Fairness: {best_F[knee_idx, 0]:.4f}")
print(f"  Excitement: {best_F[knee_idx, 1]:.4f}")

---

## 7. Visualisierung

In [ ]:
# Plot: Alle Runs + Kombinierte Front
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Links: Alle einzelnen Runs
colors = plt.cm.viridis(np.linspace(0, 1, N_SEEDS))
for i, (F, run) in enumerate(zip(all_F, all_runs)):
    axes[0].scatter(F[:, 0], F[:, 1], c=[colors[i]], alpha=0.5, s=20,
                   label=f"Seed {run['seed']} (HV={run['hypervolume']:.3f})")

axes[0].set_xlabel('Fairness (Win Rate)')
axes[0].set_ylabel('Excitement (Trick Changes)')
axes[0].set_title(f'All {N_SEEDS} Independent Runs')
axes[0].legend(fontsize=8, loc='lower left')
axes[0].grid(True, alpha=0.3)

# Rechts: Kombinierte Front
axes[1].scatter(best_F[:, 0], best_F[:, 1], c='blue', s=50, alpha=0.7, label='Pareto-Front')

# Extrempunkte markieren
axes[1].scatter(best_F[best_fairness_idx, 0], best_F[best_fairness_idx, 1], 
               c='green', s=150, marker='*', label='Best Fairness', zorder=5)
axes[1].scatter(best_F[best_excitement_idx, 0], best_F[best_excitement_idx, 1], 
               c='red', s=150, marker='*', label='Best Excitement', zorder=5)
axes[1].scatter(best_F[knee_idx, 0], best_F[knee_idx, 1], 
               c='orange', s=150, marker='*', label='Knee-Point', zorder=5)

axes[1].set_xlabel('Fairness (Win Rate)')
axes[1].set_ylabel('Excitement (Trick Changes)')
axes[1].set_title(f'Combined Pareto-Front (HV={combined_hv:.4f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'final_optimization_fronts.png', dpi=150)
plt.show()

In [ ]:
# Hypervolume-Verteilung
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(range(N_SEEDS), hvs, color='steelblue', alpha=0.7)
ax.axhline(np.mean(hvs), color='red', linestyle='--', label=f'Mean: {np.mean(hvs):.4f}')
ax.axhline(combined_hv, color='green', linestyle='-', label=f'Combined: {combined_hv:.4f}')

ax.set_xlabel('Run')
ax.set_ylabel('Hypervolume')
ax.set_title('Hypervolume per Run')
ax.set_xticks(range(N_SEEDS))
ax.set_xticklabels([f'Seed {s}' for s in FINAL_SEEDS], rotation=45)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'final_optimization_hv.png', dpi=150)
plt.show()

---

## 8. Ergebnisse speichern

In [ ]:
# Alle Ergebnisse speichern
final_results = {
    'config': {
        'algorithm': FINAL_ALGORITHM,
        'hyperparameters': {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) 
                           for k, v in algo_params.items() 
                           if k not in ['hypervolume', 'hypervolume_mean', 'hypervolume_std', 'status', 'trial_id', 'algorithm']},
        'R': R_FINAL,
        'generations': N_GEN_FINAL,
        'n_seeds': N_SEEDS,
        'K': K,
        'L': L,
        'total_runtime_minutes': total_time / 60
    },
    'statistics': {
        'hypervolume_mean': float(np.mean(hvs)),
        'hypervolume_std': float(np.std(hvs)),
        'hypervolume_min': float(np.min(hvs)),
        'hypervolume_max': float(np.max(hvs)),
        'combined_hypervolume': float(combined_hv),
        'combined_n_solutions': len(best_F)
    },
    'runs': [{k: v for k, v in r.items() if k not in ['F', 'X']} for r in all_runs],
    'extremes': {
        'best_fairness': {'fairness': float(best_F[best_fairness_idx, 0]), 
                          'excitement': float(best_F[best_fairness_idx, 1])},
        'best_excitement': {'fairness': float(best_F[best_excitement_idx, 0]), 
                            'excitement': float(best_F[best_excitement_idx, 1])},
        'knee_point': {'fairness': float(best_F[knee_idx, 0]), 
                       'excitement': float(best_F[knee_idx, 1])}
    }
}

with open(RESULTS_DIR / 'final_optimization_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print(f"Results saved to: {RESULTS_DIR / 'final_optimization_results.json'}")

In [ ]:
# Pareto-Front speichern
np.save(RESULTS_DIR / 'final_pareto_front_F.npy', best_F)
np.save(RESULTS_DIR / 'final_pareto_front_X.npy', best_X)

# Als CSV für einfachen Zugriff
front_df = pd.DataFrame(best_F, columns=['Fairness', 'Excitement'])
front_df.to_csv(RESULTS_DIR / 'final_pareto_front.csv', index=False)

print(f"Pareto-Front saved:")
print(f"  - {RESULTS_DIR / 'final_pareto_front_F.npy'}")
print(f"  - {RESULTS_DIR / 'final_pareto_front_X.npy'}")
print(f"  - {RESULTS_DIR / 'final_pareto_front.csv'}")

In [ ]:
# Ausgewählte Decks speichern (Extrempunkte + Knee-Point)
selected_decks = {
    'fairness_max': {
        'deck': best_X[best_fairness_idx].tolist(),
        'fairness': float(best_F[best_fairness_idx, 0]),
        'excitement': float(best_F[best_fairness_idx, 1])
    },
    'excitement_max': {
        'deck': best_X[best_excitement_idx].tolist(),
        'fairness': float(best_F[best_excitement_idx, 0]),
        'excitement': float(best_F[best_excitement_idx, 1])
    },
    'knee_point': {
        'deck': best_X[knee_idx].tolist(),
        'fairness': float(best_F[knee_idx, 0]),
        'excitement': float(best_F[knee_idx, 1])
    }
}

with open(RESULTS_DIR / 'final_selected_decks.json', 'w') as f:
    json.dump(selected_decks, f, indent=2)

print(f"Selected decks saved to: {RESULTS_DIR / 'final_selected_decks.json'}")

---

## 9. Zusammenfassung

In [ ]:
print("\n" + "=" * 70)
print("FINALE OPTIMIERUNG - ZUSAMMENFASSUNG")
print("=" * 70)
print(f"\nAlgorithmus: {FINAL_ALGORITHM}")
print(f"Konfiguration: R={R_FINAL}, Gen={N_GEN_FINAL}, Seeds={N_SEEDS}")
print(f"\nHypervolume: {np.mean(hvs):.4f} ± {np.std(hvs):.4f}")
print(f"Kombinierte Front: {combined_hv:.4f} ({len(best_F)} Lösungen)")
print(f"\nGesamtlaufzeit: {total_time/60:.1f} Minuten")
print("\n" + "=" * 70)